# 04 · Temporal Fusion Transformer

Own TFT implementation (`energy_forecast.tft`): variable selection networks, LSTM encoder/decoder, interpretable multi-head attention, quantile output (10/50/90 %).
168 h encoder, 25 h decoder starting at 00:00 of the delivery day, monthly refit on a 2-year window with early stopping on the last 30 days.

Variants differ in the known covariates: calendar only (`tft`), + TSO forecasts (`tft_cov`), + engineered residual-load features (`tft_eng`), + weather and fuel prices (`tft_all`).

Run first: `uv run python scripts/benchmark.py --start ... --end ... --models naive_24h lightgbm chronos2_cov ensemble tft tft_cov tft_eng tft_all`

In [1]:
from datetime import date

import plotly.io as pio
import polars as pl

from energy_forecast import evaluate, plots
from energy_forecast.data import ROOT, TEST, load

pio.templates.default = "plotly_white"
start, end = TEST
name = f"{start}_{end}"
df = load()
preds = pl.read_parquet(ROOT / f"data/processed/benchmark_{name}.parquet")
tft = pl.read_parquet(ROOT / f"data/processed/tft_{name}.parquet")
vsn = pl.read_parquet(ROOT / f"data/processed/tft_{name}_vsn.parquet")
lags = pl.read_parquet(ROOT / f"data/processed/tft_{name}_attention.parquet")

## Accuracy and significance

In [2]:
pl.read_csv(ROOT / f"results/scores_{name}.csv")

model,mae,rmse,bias,hours
str,f64,f64,f64,i64
"""chronos2_cov""",12.590492,23.928925,-0.615855,8760
"""ensemble""",12.591133,23.082328,-1.854919,8760
"""lightgbm""",14.646479,24.933225,-3.093984,8760
"""tft_eng""",15.221608,26.975985,-0.381264,8760
"""naive_24h""",31.261788,48.844408,-0.433352,8760


In [3]:
models = pl.read_csv(ROOT / f"results/scores_{name}.csv")["model"].to_list()
plots.dm_heatmap(evaluate.dm_matrix(df, preds, models)).update_layout(height=600)

In [4]:
plots.mae_by(df, preds, "month")

## Quantile calibration

Share of hours inside the 80 % interval (target 0.80) and pinball loss.

In [5]:
tft.join(df.select("time", "price"), on="time").group_by("model").agg(
    coverage=((pl.col("price") >= pl.col("q10")) & (pl.col("price") <= pl.col("q90"))).mean(),
    width=(pl.col("q90") - pl.col("q10")).mean(),
).sort("model")

model,coverage,width
str,f64,f64
"""tft_eng""",0.721005,37.660461


## Variable selection

Mean softmax weight per input in the encoder (past) and decoder (future).

In [6]:
for model in vsn["model"].unique().sort():
    plots.tft_importance(vsn, model).show()

## Attention

Where the decoder looks: peaks at −24 h and −168 h indicate daily and weekly patterns.

In [7]:
plots.attention_by_lag(lags)

In [8]:
plots.forecast_lines(
    df,
    preds.filter(pl.col("model").is_in(["chronos2_cov", "lightgbm", "tft_eng"])),
    date(2026, 6, 15),
    date(2026, 6, 21),
)